In [ ]:
# !wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
!curl -L -O https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

In [ ]:
!pip install llama-cpp-python

In [ ]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1, # indicates that all layers must be using GPU's
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [ ]:
# Generate a conversation and ask a basic question
llm.invoke( "What is 1 + 1?")

In [ ]:
# from langchain import PromptTemplate
from langchain_core.prompts import PromptTemplate

# create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables =["input_prompt"]
)

In [ ]:
basic_chain = prompt | llm

# MEMORY

In [ ]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt":"Hii | my name is Rajia. what is 1+1",
    }
)

In [ ]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "what is my name ?"})

## Conversation Buffer
Simply remind the LLMs what has happened in the past

In [ ]:
from langchain_core.prompts import PromptTemplate
template = """ <s><|user|>
Current conversation: {chat_history}
{input_prompt}<|end|>
<|assistant|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain import LLMChain

# Define the type of Memory wwe will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm, 
    memory=memory
)

In [ ]:
# Next, we ask the LLM to reproduce the name
llm_chain.invoke({"input_prompt": "what is my name ?"})

# Conversation Buffer Windowed Memory
Use last 'K' conversations instead of maintaing full chat history. </br>
In Langchain, we can use Conversation BUffer Windowed Memory to decide how many conversations are passes to the input prompt.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm, 
    memory=memory
)

In [ ]:
# Next, we ask the LLM to reproduce the name
llm_chain.invoke({"input_prompt": "what is my Rajni and I am 30 years old. What is 1+1 ?"})
llm_chain.invoke({"input_prompt": "what is 3 + 3?"})

In [ ]:
llm_chain.invoke({"input_prompt": "what is my name ?"})

In [ ]:
llm_chain.invoke({"input_prompt": "what is my age ?"})

# Conversation SUmmary
Summarizes entire conversation hostory into key points. The summarization process is done by another LLM.

In [ ]:
# Create a summary prompt template
summary_prompt_template = """ <s><|user|> Summarize the conversations and update with the new lines.

Current summary :
{summary}

new lines of conversation:
{new_lines}

New summary : <|end|?
<|assistant|>
"""

summary_prompt = PromptTemplate(
    input_variables = ["new_lines", "summary"],
    template=summary_prompt_template
)

In [ ]:
from langchain.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm, 
    memory_key = "chat_history",
    prompt = summary_prompt
)

llm_chain = LLMChain(
    prompt=prompt,
    llm=llm, 
    memory=memory
)

In [ ]:
# Next, we ask the LLM to reproduce the name
llm_chain.invoke({"input_prompt": "what is my Rajni and I am 30 years old. What is 1+1 ?"})
llm_chain.invoke({"input_prompt": "what is 3 + 3?"})